# Derive annual employment fluctuations for parametrising regional constraints
Felix Zaussinger | 10.11.2022

**Core Analysis Goal(s)**
1. Read country-, region-, and occupation-specific employment numbers for the last 10-15 years.
2. Analyse yearly changes, mean/sd of timeseries
3. Use to parametrise the regional constraint in the job transition simulations

**Key Insight(s)**
1.
2.
3.

In [137]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils, plotting_utils, stats_utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("notebook")
# sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)
pd.options.display.float_format = "{:,.2f}".format

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read data

In [138]:
countries = [
    "AT",
    "BE",
    "CH",
    #"CY",
    "CZ",
    "DE",
    "DK",
    #"EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IS",
    "IT",
    "LT",
    #"LU",
    #"LV",
    #"NL",
    "NO",
    "PT",
    "RO",
    "SE",
    "SK",
    "UK",
]

years = np.arange(1998, 2020)
years

array([1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019])

In [139]:
from src.data.lfs import EuLfs

config = utils.load_config(os.path.join(useful_paths.config_dir, "eu_lfs_config.yml"))
eulfs = EuLfs(config=config)

##### Create data cube

In [140]:
from tqdm import tqdm

stacked_data = {}
stacked_stats = {}

cols = ["REFYEAR", "COUNTRYW", "REGIONW", "ISCO3D", "COEFF"]
for country in tqdm(countries):
    for year in years:
        try:
            df, processing_stats = eulfs.read_filtered_file(country=country, year=year)

            stacked_data[(country, year)] = df[cols]
            stacked_stats[(country, year)] = processing_stats

        # continue if country-year file does not exist
        except FileNotFoundError as e:
            print(e)
            continue

# combine
df_stacked_data = pd.concat(list(stacked_data.values()), axis=0)

  0%|          | 0/22 [00:03<?, ?it/s]


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
df_stacked_data.loc[df_stacked_data["REFYEAR"] == "2019"]

In [ ]:
# sums for 2019
df_stacked_data_2019 = df_stacked_data.loc[df_stacked_data["REFYEAR"] == "2019"].groupby(["COUNTRYW", "REGIONW", "ISCO3D"]).sum()

##### Derive statistics about yearly employment changes

In [ ]:
agg_dict = {"COEFF": ["min", "max", "mean", "std", "median"]}
fluctuations_over_time = df_stacked_data.groupby(["COUNTRYW", "REGIONW", "ISCO3D", "REFYEAR"]).aggregate({"COEFF": "sum"})

# YoY changes
fluctuations_over_time_YoY = fluctuations_over_time - fluctuations_over_time.groupby(["COUNTRYW", "REGIONW", "ISCO3D"]).shift(1)

# keep only yearly changes that are positive (employment influx)
fluctuations_over_time_YoY[fluctuations_over_time_YoY < 0] = np.nan

# calculate stats over time
fluctuations_stats_abs = fluctuations_over_time_YoY.reset_index().groupby(["COUNTRYW", "REGIONW", "ISCO3D"]).aggregate(agg_dict)
fluctuations_stats_abs.columns = ['_'.join(col).strip() for col in fluctuations_stats_abs.columns.values]

# calculate a relative variant
fluctuations_stats_rel = fluctuations_stats_abs.copy(deep=True)
fluctuations_stats_rel["COEFF_2019"] = df_stacked_data_2019
for col in fluctuations_stats_rel.columns:
    fluctuations_stats_rel[col] = fluctuations_stats_rel[col].divide(fluctuations_stats_rel["COEFF_2019"])

# add 2019 data as reference
fluctuations_stats_abs["COEFF_2019"] = df_stacked_data_2019

In [ ]:
fluctuations_stats_abs

In [ ]:
fluctuations_stats_rel

##### YoY relative changes in employment
- on average, one ISCO-3D occupation in a NUTS-2 region can absorb up to 43% of its current employment in a year (!!!)

In [ ]:
fluctuations_stats_rel.dropna().describe()

In [ ]:
ax = fluctuations_stats_rel.plot.box()
ax.set_ylim(-0.1, 1.5)
plt.xticks(rotation=90)
plt.ylabel("YoY relative employment changes [-]")
plt.tight_layout()
plt.savefig(os.path.join(useful_paths.figure_dir, "03_eulfs", "lfs_yoy_employment_fluctuations_stats_rel.png"), dpi=300)

##### Save

In [ ]:
utils.save_df_to_files(
    df=fluctuations_stats_abs,
    output_dir=config["paths"]["processed"],
    fname_no_ext="lfs_employment_fluctuations_abs_{}_{}".format(years[0], years[-1])
)

utils.save_df_to_files(
    df=fluctuations_stats_rel,
    output_dir=config["paths"]["processed"],
    fname_no_ext="lfs_employment_fluctuations_rel_{}_{}".format(years[0], years[-1])
)